# HAB-P10 on the GPU: evolved feelings, 24 brains trained at once

24 simulated **Unitree Go2** robots, each with its own brain and its own "feelings" about its body (how much
motor heat bothers it and from what temperature, what effort costs, whether effort hurts more when hot,
whether it enjoys moving). All 24 brains learn **simultaneously in one compiled GPU program**; between
rounds, selection copies the best brains over the worst. Three groups of 8:

* **evolved**: feelings are inherited from the winners and mutate
* **random**: random feelings, never inherited (variety without evolution)
* **none**: no feelings

Same design, genomes and pre-registered analysis as the CPU experiment
([`docs/prereg/HABP10_evolved_feelings.md`](https://github.com/pablopedrosap/homeostatic-Alife/blob/main/docs/prereg/HABP10_evolved_feelings.md)),
with several times more training per robot.

**How to run:** Session options → **GPU T4 x2**, **Internet on** → **Save Version → Save & Run All**
(~10 h; you can close the tab). Results: the version's **Output** tab → `results/habp10_gpu/`.

In [ ]:
# 1) Setup — exact versions tested together (do not change them)
import os
os.environ["MUJOCO_GL"] = "egl"                       # headless rendering on the GPU server
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false" # leave GPU memory for the video renderer
!pip install -q "jax[cuda12]==0.9.2" "brax==0.14.2" "mujoco==4.0.0" "mujoco-mjx==4.0.0" mediapy

In [ ]:
# 2) Check that JAX sees the GPU
import jax
print(jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU! Kaggle: right panel → Session options → Accelerator → GPU T4 x2"


In [ ]:
# 3) Get the project code and the Go2 robot model
import sys, warnings
warnings.simplefilter("ignore")
!rm -rf homeostatic-Alife && git clone -q --depth 1 https://github.com/pablopedrosap/homeostatic-Alife.git
sys.path.insert(0, "homeostatic-Alife/kaggle")
import go2_body_budget as G
xml = G.fetch_go2("mujoco_menagerie")
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(OUT, exist_ok=True)
print("robot:", xml, "| results go to:", OUT)

## Quick check (~20 min, not part of the experiment)
A small population (4 brains per group) trains 1.2M steps per brain. If the robots are not starting to
walk, the notebook stops here instead of spending 10 hours of GPU. If it stops, copy the output to the chat.

In [ ]:
import numpy as np
import population as P
qc = P.Population(xml, P.PopConfig(per_arm=4, rounds=1, steps_per_round=1_200_000, mission_s=20.0, final_lifetimes=1),
                  out_dir=f"{OUT}/quick_check")
qc.run()
med = float(np.median([h["fitness"] for h in qc.history]))
print(f"QUICK CHECK: median distance in 4 x 20-s missions = {med:.1f} m (needs >= 16 m)")
assert med >= 16, "Robots are not learning to walk: copy this output to the chat."
print("QUICK CHECK OK")
del qc

## The run
`hours_budget` sizes the training to the time you give it: after the first minutes it measures the speed
and chooses the steps per brain per round. Progress lines show each group's mean walking speed.

In [ ]:
import population as P
cfg = P.PopConfig(hours_budget=10.0)          # 3 groups x 8 brains, 12 rounds
pop = P.Population(xml, cfg, out_dir=f"{OUT}/habp10_gpu")
pop.run()

## Pre-registered analysis (the same script as the CPU run)

In [ ]:
!python homeostatic-Alife/scripts/analyze_go2_v10.py --dir {OUT}/habp10_gpu --fig-dir {OUT}/habp10_gpu
from IPython.display import Image
Image(f"{OUT}/habp10_gpu/habp10_feelings.png")

## Video: the best robot of each group carrying 3 kg

In [ ]:
import mediapy
last = [h for h in pop.history if h["round"] == cfg.rounds - 1]
for arm in P.ARMS:
    best = max((h for h in last if h["arm"] == arm), key=lambda h: h["fitness"])
    frames = G.render(xml, pop.record(best["id"], payload_kg=3.0, seconds=20.0))
    mediapy.write_video(f"{OUT}/habp10_gpu/{arm}_best_3kg.mp4", frames, fps=25)
    print(arm, "robot", best["id"]); mediapy.show_video(frames, fps=25)

## Results
Download `results/habp10_gpu/` (Output tab) and bring it back to the repository: `final_lifetimes.csv`,
`history.json`, `final_members.json`, `run_config.json`, `verdict.txt`, the figure and the videos.